In [1]:
from data_collection.data_adapters import DatabentoGenericAdapter
from dotenv import load_dotenv
from pathlib import Path
import os
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
load_dotenv()
pl.enable_string_cache()

CACHE_DIR = Path.cwd().resolve().parents[0] / "data" / "dbn_cache" / "generic"
adapter = DatabentoGenericAdapter(os.getenv("DATABENTO-API-KEY"), CACHE_DIR)

print(f"CACHE_DIR: {CACHE_DIR}")
# Clear the cache if needed
# adapter.clear_cache()

with open("../top_50_universe.txt", "r") as f:
    ACTIVE_SET = [s for s in [line.strip() for line in f] if s not in ['CRCL', 'OMDA']]
START = "2023-06-01"
END = "2025-06-01"
DATASET = "XNAS.ITCH"

df_bars = (
    adapter.scan_data(
        tickers="ALL_SYMBOLS",
        start_date=START,
        end_date=END,
        dataset=DATASET,
        schema="ohlcv-1d"
    )
    .select(["ts_event", "symbol", "close"])
    .with_columns(
        pl.col("symbol").cast(pl.Categorical)
    )
    .filter(
        pl.col("symbol").is_in(ACTIVE_SET)
    )
    .sort(["ts_event", "symbol"])
)

CACHE_DIR: /Users/michaelcusack-nelkin/GitHub/Summer_Research_2026/data/dbn_cache/generic
[GenericAdapter] scan_parquet: 548cfb7cc98703d7d8a8a14fe79dfe1a.parquet


In [2]:
CAX = (
     pl.read_csv("../CAX/corporate_actions.csv")
     .sort(["date", "symbol"])
     .with_columns(
         pl.col("date").str.to_datetime(time_unit="ns", time_zone="UTC"),
         pl.col("symbol").cast(pl.Categorical)
     )
)
CAX

symbol,date,dividend,split
cat,"datetime[ns, UTC]",f64,f64
"""AGG""",2023-06-01 00:00:00 UTC,0.258,0.0
"""AJG""",2023-06-01 00:00:00 UTC,0.55,0.0
"""ALL""",2023-06-01 00:00:00 UTC,0.89,0.0
"""ANGL""",2023-06-01 00:00:00 UTC,0.131,0.0
"""BAC""",2023-06-01 00:00:00 UTC,0.22,0.0
…,…,…,…
"""FIS""",2026-06-11 00:00:00 UTC,0.44,0.0
"""FNV""",2026-06-11 00:00:00 UTC,0.44,0.0
"""JXN""",2026-06-11 00:00:00 UTC,0.9,0.0


In [3]:
daily_px = (
      df_bars
      .collect(engine="streaming")
    #   .filter(pl.col("symbol").is_in(ACTIVE_SET))
      .join(CAX, left_on=["ts_event", "symbol"], right_on=["date", "symbol"], how="left")
      .sort(["symbol", "ts_event"])
      .with_columns(
          pl.col("dividend").fill_null(0.0),
          # 0.0 / null == "no split" -> ratio of 1.0, not 0.0
          pl.when(pl.col("split").fill_null(0.0) == 0.0)
            .then(1.0).otherwise(pl.col("split")).alias("split"),
      )
      .with_columns(
          (pl.col("split") / (1 - pl.col("dividend") / pl.col("close").shift().over("symbol"))).alias("CAX")
      )
      .with_columns(pl.col("CAX").fill_null(1.0))            # first row per symbol: no prev_close
      .with_columns(pl.col("CAX").cum_prod().over("symbol"))
  )
daily_px

ts_event,symbol,close,dividend,split,CAX
"datetime[ns, UTC]",cat,f64,f64,f64,f64
2023-06-01 00:00:00 UTC,"""AAPL""",180.58,0.0,1.0,1.0
2023-06-02 00:00:00 UTC,"""AAPL""",181.1,0.0,1.0,1.0
2023-06-05 00:00:00 UTC,"""AAPL""",179.12,0.0,1.0,1.0
2023-06-06 00:00:00 UTC,"""AAPL""",179.04,0.0,1.0,1.0
2023-06-07 00:00:00 UTC,"""AAPL""",177.9,0.0,1.0,1.0
…,…,…,…,…,…
2025-05-23 00:00:00 UTC,"""VOO""",532.1,0.0,1.0,1.028759
2025-05-27 00:00:00 UTC,"""VOO""",543.53,0.0,1.0,1.028759
2025-05-28 00:00:00 UTC,"""VOO""",548.37,0.0,1.0,1.028759


In [4]:
adj_px = (
    daily_px
    .with_columns((pl.col("close") * pl.col("CAX")).alias("adj_close"))
    .select(["ts_event", "symbol", "adj_close"])
)
adj_px

ts_event,symbol,adj_close
"datetime[ns, UTC]",cat,f64
2023-06-01 00:00:00 UTC,"""AAPL""",180.58
2023-06-02 00:00:00 UTC,"""AAPL""",181.1
2023-06-05 00:00:00 UTC,"""AAPL""",179.12
2023-06-06 00:00:00 UTC,"""AAPL""",179.04
2023-06-07 00:00:00 UTC,"""AAPL""",177.9
…,…,…
2025-05-23 00:00:00 UTC,"""VOO""",547.402684
2025-05-27 00:00:00 UTC,"""VOO""",559.1614
2025-05-28 00:00:00 UTC,"""VOO""",564.140593


In [5]:
daily_ret = (
    adj_px
    .with_columns(
        pl.col("adj_close").pct_change().over("symbol").alias("ret")
    )
    .pivot(
        on="symbol",
        on_columns=ACTIVE_SET,
        index="ts_event",
        values="ret"
    )
    .with_columns(
        pl.all().exclude("ts_event").fill_null(pl.all().exclude("ts_event").mean())
    )
    .with_columns(
        (pl.all().exclude("ts_event") - pl.all().exclude("ts_event").mean()).round(6)
    )
)
daily_ret

ts_event,NVDA,TSLA,SPY,QQQ,AAPL,MSFT,AMZN,META,GOOGL,AVGO,PLTR,MSTR,GOOG,NFLX,TQQQ,TLT,AMD,IWM,COST,COIN,APP,MU,IBIT,SMCI,QCOM,TXN,CRWD,CSCO,INTC,HOOD,ADBE,SQQQ,BKNG,MRVL,AMAT,TMUS,PEP,SMH,LIN,ASML,INTU,BABA,TSM,ISRG,LRCX,CRWV,UNH,PDD,VOO
"datetime[ns, UTC]",f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2023-06-01 00:00:00 UTC,-0.0,-0.0,0.0,0.0,0.0,0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,-0.0,-0.0,-0.0,-0.0,0.0,-0.0,0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2023-06-02 00:00:00 UTC,-0.021408,0.026902,0.013491,0.006259,0.00249,0.004946,0.006565,-0.004621,0.007726,0.043334,-0.008817,-0.000188,0.007765,-0.011219,0.018855,-0.012055,-0.020249,0.036361,0.00627,0.014123,0.004703,0.001497,-0.0,-0.042313,-0.003653,-0.003304,-0.044435,0.003118,0.008344,0.023769,0.023035,-0.029928,0.024091,-0.0215,0.000356,-0.063681,0.008266,-0.003734,0.002476,-0.001746,0.01907,0.016906,-0.00405,0.006028,-0.013176,0.0,0.012812,0.003619,0.013587
2023-06-05 00:00:00 UTC,-0.009008,0.008421,-0.003998,-0.002275,-0.011323,0.000288,0.006263,-0.008301,0.00974,-0.014004,0.054365,-0.086325,0.010169,0.003448,-0.006002,0.00095,0.001576,-0.015992,0.008982,-0.098401,0.034828,-0.023388,-0.0,-0.004415,-0.027608,-0.032469,0.012681,0.002703,-0.053782,-0.033898,-0.006,-0.002291,0.019152,-0.040247,-0.014688,0.008408,0.00494,-0.014893,0.012776,-0.007562,0.024533,0.000342,-0.011245,0.014036,-0.014585,0.0,-0.004914,-0.024278,-0.004262
2023-06-06 00:00:00 UTC,-0.008403,0.03989,0.003539,0.001448,-0.000836,-0.007403,0.010126,0.00153,0.009549,-0.021866,0.00906,0.072786,0.009495,-0.009208,0.004334,0.007228,0.059541,0.028644,-0.003343,-0.09896,-0.10432,0.000812,-0.0,0.058019,0.031883,0.001779,-0.00234,-0.00807,0.041037,0.002806,-0.002603,-0.013789,0.011442,0.029326,0.00315,-0.023548,-0.015178,0.005522,-0.008135,-0.006378,0.010354,0.027102,0.020153,-0.006676,0.008665,0.0,-0.017837,0.07224,0.003712
2023-06-07 00:00:00 UTC,-0.029175,-0.010576,-0.005115,-0.018914,-0.006757,-0.032833,-0.041243,-0.034214,-0.037731,0.003366,-0.065435,-0.040639,-0.039038,-0.002475,-0.056331,-0.015617,-0.054513,0.018387,-0.012039,0.001993,-0.083974,-0.006868,-0.0,0.011281,-0.010701,0.012303,-0.031776,-0.001098,0.009376,0.015905,-0.034265,0.047089,-0.028892,-0.002328,0.009274,-0.014644,-0.008878,-0.005824,0.005726,-0.009193,-0.056721,-0.018356,-0.002326,-0.027957,-0.000493,0.0,-0.011014,-0.013071,-0.005452
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2025-05-23 00:00:00 UTC,-0.015441,-0.008513,-0.007942,-0.011577,-0.028189,-0.008497,-0.013214,-0.019099,-0.021767,-0.014884,-0.002438,-0.097906,-0.022134,-0.006077,-0.036348,0.001824,-0.006863,-0.004444,-0.01245,-0.047098,-0.000818,-0.021052,-0.042376,-0.037392,-0.016746,-0.025554,0.022936,-0.005596,-0.024931,-0.034908,-0.014713,0.024894,-0.003628,-0.019242,-0.017609,0.005508,-0.003983,-0.019389,0.004521,-0.016139,-0.000757,-0.00561,-0.023784,-0.002517,-0.02091,-0.013871,-0.006776,-0.006407,-0.007925
2025-05-27 00:00:00 UTC,0.030599,0.079291,0.020521,0.024597,0.022946,0.021594,0.026721,0.023434,0.027281,0.033191,-0.007782,0.007697,0.027433,0.020325,0.075327,0.014006,0.040929,0.025915,0.009064,0.013757,0.070302,0.036335,0.012547,0.054583,0.02607,0.04279,0.033813,0.012646,0.024738,0.050193,0.012888,-0.081885,0.018969,0.058837,0.027462,0.004926,0.015082,0.03435,0.01766,0.039559,0.040965,-0.010842,0.031846,0.01917,0.03864,0.150917,-0.001844,-0.138727,0.020703
2025-05-28 00:00:00 UTC,0.039684,-0.009658,0.008138,0.012685,0.034838,0.001668,0.02766,0.016473,0.011467,0.045791,0.015875,-0.010434,0.009882,-0.005769,0.037645,-0.005553,0.005272,0.007,-0.002006,-0.030802,0.046352,0.028762,-0.013192,0.02299,0.009259,0.01507,0.000065,-0.006182,0.002269,-0.000852,0.004948,-0.048886,0.010328,0.052853,0.017

In [6]:
# causal_vol = daily_ret.with_columns(pl.all().exclude("ts_event").pow(2).ewm_mean(alpha=0.06, adjust=True).sqrt().shift(1))
# st_ret = daily_ret.select(pl.all().exclude("ts_event")) / causal_vol.select(pl.all().exclude("ts_event"))
# st_ret


# floor the vol so a collapsed forecast can't produce a divide-by-~0
# (cross-sectional: e.g. 1st percentile of that day's vols, or a small abs floor)
causal_vol = daily_ret.with_columns(
    pl.all().exclude("ts_event").pow(2).ewm_mean(alpha=0.06, adjust=True).sqrt().shift(1)
)
VOL_FLOOR = 1e-3   # ~0.1%/day
st_ret = (
    daily_ret.select(pl.all().exclude("ts_event"))
    / causal_vol.select(pl.all().exclude("ts_event")).select(pl.all().clip(lower_bound=VOL_FLOOR))
).select(pl.all().clip(-10, 10))   # a >10σ standardized daily return is a data artifact

st_ret


NVDA,TSLA,SPY,QQQ,AAPL,MSFT,AMZN,META,GOOGL,AVGO,PLTR,MSTR,GOOG,NFLX,TQQQ,TLT,AMD,IWM,COST,COIN,APP,MU,IBIT,SMCI,QCOM,TXN,CRWD,CSCO,INTC,HOOD,ADBE,SQQQ,BKNG,MRVL,AMAT,TMUS,PEP,SMH,LIN,ASML,INTU,BABA,TSM,ISRG,LRCX,CRWV,UNH,PDD,VOO
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
-10.0,10.0,10.0,6.259,2.49,4.946,6.565,-4.621,7.726,10.0,-8.817,-0.188,7.765,-10.0,10.0,-10.0,-10.0,10.0,6.27,10.0,4.703,1.497,-0.0,-10.0,-3.653,-3.304,-10.0,3.118,8.344,10.0,10.0,-10.0,10.0,-10.0,0.356,-10.0,8.266,-3.734,2.476,-1.746,10.0,10.0,-4.05,6.028,-10.0,0.0,10.0,3.619,10.0
-0.586075,0.435993,-0.412762,-0.506264,-6.333781,0.081103,1.328766,-2.502046,1.755922,-0.450116,8.588146,-10.0,1.824054,0.428069,-0.443374,0.109763,0.108406,-0.612587,1.995292,-9.704506,10.0,-10.0,-0.0,-0.145331,-10.0,-10.0,0.397493,1.207455,-8.977668,-1.986388,-0.362797,-0.106622,1.107287,-2.60733,-10.0,0.183901,0.832401,-5.555316,7.186958,-6.032444,1.791847,0.028176,-3.867277,3.243179,-1.541785,0.0,-0.534219,-9.343836,-0.436909
-0.624055,2.445601,0.43479,0.375443,-0.121337,-2.589461,1.905485,0.272552,1.305882,-0.829662,0.276635,1.416811,1.261014,-1.355991,0.378503,1.035759,5.079902,1.243372,-0.517938,-1.673776,-4.99057,0.058228,-0.0,2.362836,1.924781,0.091623,-0.087556,-3.343956,1.267891,0.115031,-0.189143,-0.796055,0.636535,1.087223,0.360271,-0.635026,-2.709093,0.60541,-1.05155,-1.38303,0.566356,2.777817,2.843142,-0.73783,0.750977,0.0,-2.243728,4.948545,0.45051
-2.374099,-0.421797,-0.712416,-5.607284,-1.147544,-7.175981,-5.917776,-7.053381,-4.724116,0.133538,-2.311245,-0.700386,-4.810636,-0.328699,-5.622668,-2.215924,-1.666557,0.744459,-2.085711,0.027588,-1.462794,-0.577506,-0.0,0.305997,-0.489721,0.742308,-1.393067,-0.233817,0.268242,0.763154,-2.901946,2.865937,-1.756619,-0.084235,1.215202,-0.431731,-0.958217,-0.702288,0.729673,-1.783143,-3.438597,-1.116803,-0.19143,-3.302533,-0.04553,0.0,-0.955142,-0.328534,-0.74831
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
-0.400951,-0.185018,-0.445083,-0.537645,-0.997228,-0.414814,-0.435369,-0.544211,-0.777881,-0.401753,-0.047433,-2.020114,-0.787855,-0.267392,-0.558795,0.169457,-0.173723,-0.228726,-0.861658,-0.828469,-0.014069,-0.481423,-1.740198,-0.499992,-0.579086,-0.799101,0.667921,-0.309043,-0.69326,-0.56203,-0.770736,0.379933,-0.15557,-0.410263,-0.525021,0.27871,-0.296493,-0.6196,0.251096,-0.602693,-0.026589,-0.152335,-0.808631,-0.098666,-0.577629,-0.14067,-0.115812,-0.203911,-0.438352
0.815346,1.775482,1.178739,1.167476,0.811884,1.081397,0.902611,0.682295,0.986693,0.919328,-0.15615,0.1459,0.987776,0.920316,1.182697,1.34087,1.067559,1.373426,0.632218,0.2443,1.247109,0.850758,0.486498,0.746862,0.920047,1.352838,1.001452,0.718142,0.698869,0.825237,0.683509,-1.283102,0.83831,1.286999,0.837187,0.256457,1.154739,1.118567,1.009625,1.506341,1.484057,-0.303432,1.094152,0.774835,1.089412,1.57759,-0.032493,-4.547854,1.173941
1.068223,-0.203519,0.462084,0.595631,1.245452,0.08311,0.939567,0.487502,0.415065,1.274257,0.328294,-0.203857,0.356079,-0.262428,0.584111,-0.519332,0.136938,0.361494,-0.142509,-0.563104,0.808888,0.679092,-0.523638,0.318831,0.328278,0.465009,0.001925,-0.356278,0.065108,-0.014144,0.266713,-0.751587,0.460558,1.133991,0.53976,-0.237116,-0.056099,0.541024,-0.222263,0.078222,0.413725,-0.271555,0.439692,0.168928,0.611042,-0.223768,0.317482,-0.594981,0.455692


In [7]:
null_columns = [
    col for col in st_ret.columns 
    if st_ret.select(pl.col(col).is_null().all()).item()
]
null_columns

[]

In [8]:
X = st_ret.drop_nulls().to_numpy()

In [9]:
# Best rank k approximation of the returns covariance matrix
U, s, Vt = np.linalg.svd(X)
k = 10
U_k, S_k, Vt_k = U[:, :k], np.diag(s[:k]), Vt[:k, :]
F = U_k
L = S_k @ Vt_k

# Residuals: what the market factor can't explain
# Adding a floor to epsilon volatility for illiquid names
eps = X - F @ L          # (T, p)
eps_var  = np.var(eps, axis=0)
EPS_FLOOR = 0.1
eps_var  = np.maximum(eps_var, EPS_FLOOR)
D = np.diag(eps_var)         # (2000, 2000) diagonal

# Factor covariance
Sigma_f = np.cov(F, rowvar=False)       # (10, 10)

# Model-implied return covariance: Σ_r = L^T Σ_f L + D
Sigma_r = L.T @ Sigma_f @ L + D         # (2000, 2000)


In [10]:
evals = np.linalg.eigvalsh(Sigma_r)

In [11]:
cond_n = np.abs(np.max(evals)) / np.abs(np.min(evals))
print(f"Condition Number: {cond_n}")

Condition Number: 239.9997418559138


In [12]:
eps_var

array([0.33283614, 0.64858673, 0.1       , 0.1       , 0.6588514 ,
       0.38315253, 0.43592656, 0.45818012, 0.1655821 , 0.47092165,
       0.53574385, 0.25736495, 0.16584488, 0.70754663, 0.1       ,
       0.60006984, 0.41989016, 0.35023166, 0.50943203, 0.2386572 ,
       0.47973948, 0.45929362, 0.44988526, 0.68014395, 0.42646564,
       0.50735966, 0.67174788, 0.34070811, 0.71773981, 0.4029652 ,
       0.86303843, 0.16178214, 0.72098499, 0.38096857, 0.28823804,
       0.58736956, 0.48713341, 0.1       , 0.68855973, 0.37276662,
       0.56024485, 0.35436498, 0.39861111, 0.62021113, 0.25801581,
       0.47633289, 0.25715916, 0.34925145, 0.10147218])

In [13]:
d = np.diag(Sigma_r)
print("diag  min/median/max:", d.min(), np.median(d), d.max())   # want median ≈ 1, max single digits
print("cond:", np.linalg.cond(Sigma_r))                          # want O(1e1–1e3), not 1e7

diag  min/median/max: 0.4882661222543069 1.3964743396869728 1.8727510644071332
cond: 239.99974185591543


In [14]:
Sigma_r - (L.T @ Sigma_f @ L + D)

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(49, 49))

In [15]:
np.save("loadings.npy", L)
np.save("factor_returns.npy", F)
np.save("idiosyncratic_diag_vec.npy", eps_var)
np.save("factor_cov_matrix.npy", Sigma_f)
np.save("cov_matrix.npy", Sigma_r)


### Modeling eps (idiosyncratic) with GARCH(1,1)

In [16]:
L[0,:]

array([ 16.1479861 ,  15.0181825 ,  23.84130197,  24.50499106,
        15.37629238,  18.65241911,  18.35974422,  16.9413321 ,
        16.82099187,  18.83534479,  11.37312315,  11.18242311,
        16.8214319 ,  11.89790515,  25.13954012,   0.63660576,
        15.57746296,  17.69093127,  12.14966147,  16.12358878,
        12.40201985,  17.77045529,   7.09020848,  11.8089253 ,
        17.78814363,  15.48419154,  12.21974782,  10.56246586,
        15.10031694,  15.17686038,  14.31312324, -16.58062489,
        15.2973742 ,  16.37131373,  19.57641897,   2.08236065,
         1.74556801,  21.6210302 ,   7.70812097,  18.88158605,
        16.78159253,  10.53242466,  17.22492651,  15.00300995,
        17.04020226,   2.04060409,   3.00042485,  10.17738407,
        23.81763868])